In [ ]:
import pandas as pd
import numpy as np
import ast
import re
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_PATH: str = 'game_dataset_cleaned.csv'
SIMILARITY_MATRIX_PATH: str = 'similarity_matrix.pkl'
GAMEINDEX_DFINDEX_PATH: str = 'gameindex_dfindex.pkl'
DFINDEX_GAMEINDEX_PATH: str = 'dfindex_gameindex.pkl'

def clean_keywords(keywords_str: str) -> str:
    keywords_list = ast.literal_eval(keywords_str)

    bad_patterns = [
            r'\bsteam\b', r'playstation', r'xbox', r'nintendo', r'wii\b', r'\bdsi\b',
            r'gameboy', r'game boy', r'apple arcade', r'google play', r'windows_store',
            r'\bpax\b', r'\be3\b', r'gamescom', r'tokyo_game_show', r'nextfest', r'game_awards',
            r'controller', r'mouse', r'keyboard', r'gamepad', r'joystick', r'touchscreen',
            r'achievement', r'trading_card', r'leaderboard', r'cloud_save', r'cross-play',
            r'digital_distribution', r'exclusive', r'downloadable_content', r'\bdlc\b',
            r'ai-generated', r'kickstarter', r'microtransaction', r'in-app', r'free-to-play',
            r'\b20[0-2][0-9]\b', r'\b19[8-9][0-9]\b',
            r'multiplayer', r'single-player', r'co-op', r'online', r'split_screen',
            r'local_multiplayer', r'voice_acting', r'remastered', r'definitive_edition'
        ]

    exact_bad_words = {
        'licensed game', 'unlicensed game', 'sequel', 'prequel', 'year in the title',
        "protagonist's name in the title", 'demo', 'free demo', 'abandonware', 'vaporware',
        'multiplayer only', 'single-player only', 'games on demand', 'greatest hits',
        'early access', 'manual included', 'physical copy protection', 'launch titles',
        'fan translation - english', 'fan translation - portuguese'
    }

    clean_list = []
    for keyword in keywords_list:
        clean_keyword = keyword.lower().strip()

        if clean_keyword in exact_bad_words:
            continue

        is_bad = any(re.search(p, clean_keyword) for p in bad_patterns)

        if not is_bad and len(clean_keyword) >= 2:
            clean_list.append(clean_keyword.replace(" ", "_"))

    return " ".join(clean_list)

def load_data() -> pd.DataFrame:
    df = pd.read_csv(DATA_PATH)
    return df;

def process_data(df: pd.DataFrame) -> pd.DataFrame:
    print("Started processing")
    def parse_string(string: str) -> str:
        items = ast.literal_eval(string)
        return ' '.join([item.lower().strip().replace(" ", "_") for item in items])

    working_df = df.dropna(subset=['genres', 'themes', 'keywords', 'platforms']).copy()
    working_df["total_score"] = working_df["aggregated_rating"].fillna(working_df["rating"]).fillna(0)
    working_df = working_df.sort_values(by="total_score", ascending=False)
    working_df = working_df.head(20_000).copy()
    working_df = working_df.reset_index(drop=True)

    print("Size", working_df.count())

    working_df["genres_str"] = working_df["genres"].apply(parse_string)
    working_df["themes_str"] = working_df["themes"].apply(parse_string)
    working_df["keywords_str"] = working_df["keywords"].apply(clean_keywords)

    working_df["content"] = ((working_df["genres_str"] + ' ') * 5 +
                             (working_df["themes_str"] + ' ') * 2 +
                             (working_df["keywords_str"] + ' ') * 10
                             )
    print("Finished processing")
    return working_df

def fit(df: pd.DataFrame) -> np.ndarray:
    print("Started training")
    tfidf = TfidfVectorizer(
        min_df=2,
        max_df=0.20,
        ngram_range=(1,1),
        stop_words="english"
        )
    tfidf_matrix = tfidf.fit_transform(df["content"]).astype(np.float32)

    cos_sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)
    print("Finished training")
    return cos_sim_matrix

def save_data(df: pd.DataFrame, similarity_matrix: np.ndarray) -> None:
    print("Started saving")
    with open(SIMILARITY_MATRIX_PATH, 'wb') as f:
        pickle.dump(similarity_matrix, f)

    game_id_to_index = {
        int(game_id) : index
        for index, game_id in enumerate(df["id"])
    }

    index_to_game_id = {
        index : int(game_id)
        for index, game_id in enumerate(df["id"])
    }

    with open(GAMEINDEX_DFINDEX_PATH, 'wb') as f:
        pickle.dump(game_id_to_index, f)
    with open(DFINDEX_GAMEINDEX_PATH, 'wb') as f:
        pickle.dump(index_to_game_id, f)
    print("Finished saving")

def train_model():
    df: pd.DataFrame = load_data()
    processed_df: pd.DataFrame = process_data(df)
    result: np.ndarray = fit(processed_df)
    save_data(processed_df, result)

train_model()



Started processing
Size id                       20000
name                     20000
category                 20000
release_date             19916
rating                   16247
aggregated_rating        10142
genres                   20000
themes                   20000
franchise                 4895
series                    9464
main_developers          18111
supporting_developers      779
publishers               18255
platforms                20000
player_perspectives      16950
game_modes               19491
game_engines              6024
similar_games            19994
keywords                 20000
cover_url                19910
summary                  20000
screenshot_urls          19213
artwork_urls             11685
total_score              20000
dtype: int64
Finished processing
Started training
Finished training
Started saving
Finished saving
